# Degree Zero — metrics notebook

Computes every metric for every translator's chain output with one code path, so models are compared with identical settings.

**Input:** every `chain_<lang>_<model>.csv` in `IN_DIR` (from `degree_zero_chain.ipynb`; your own run and any you were sent).

| block | metrics | on |
|---|---|---|
| **Intervention** (S1) | int_G = ‖e(S1) − e(G)‖, chrF vs G, noise floor from F2, **int_G(F) − int_G(L)** | L, F, F2 |
| **Round trip** (T1 vs T0) | content (e5), chrF loss, bigram loss, Burrows' Delta (several vocabulary sizes), Cosine Delta, function-word cosine, sentence length, entity / number recall; raw and normalized | B_L, B_F |
| **Ruler** | the same round-trip metrics | corpus paraphrases: ChatGPT ×1/×2/×3, Pegasus (full), Dipper |
| **Quality** (optional) | COMETKiwi QE of S1 given T0 | L, F, F2 |

Normalized = raw ÷ the same metric between unrelated texts from the same domain and source (0 = unchanged, 1 = as different as an unrelated text). Same definitions as the pilot and `round_trip_stats.py`.

**Output** (to `OUT_DIR`): `metrics_items.csv` (round trip and ruler, one row per model × item × version), `metrics_intervention.csv` (one row per model × item), `metrics_summary.json`, and `qe_scores.csv` if QE runs.

**Runtime → GPU** (any; the embeddings take a few minutes per model).

In [ ]:
!pip install -q sentence-transformers sacrebleu statsmodels
import os, re, glob, json, collections, numpy as np, pandas as pd, torch, sacrebleu
from scipy.stats import wilcoxon
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — embeddings will be slow on CPU")

## Config

In [ ]:
IN_DIR    = "/content/drive/MyDrive/degree_zero"     # where the chain_*.csv files are
OUT_DIR   = IN_DIR
EMB_MODEL = "intfloat/multilingual-e5-base"
MFW_SIZES = [50, 100, 150, 300, 500]                 # Burrows' Delta vocabulary sizes
MFW_MAIN  = 150                                      # main style metric
RANDOM_SHIFT = 7                                     # random-pair baseline: item i vs item i+7 within domain × source
INCLUDE_SMOKE = False                                # also read *_smoke.csv files
RUN_QE    = True                                     # COMETKiwi; needs HF_TOKEN in Colab Secrets + license accepted

# corpus paraphrase versions used as the ruler
PARAPHRASES = {"chatgpt×1": "chatgpt", "chatgpt×2": "chatgpt_chatgpt",
               "chatgpt×3": "chatgpt_chatgpt_chatgpt", "pegasus(full)": "pegasus(full)",
               "dipper": "dipper"}
# round-trip versions produced by the chain notebook
ROUND_TRIPS = {"T1 literal": "B_L", "T1 free": "B_F"}

## 1 · Load chain outputs

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

paths = sorted(p for p in glob.glob(f"{IN_DIR}/chain_*.csv") if INCLUDE_SMOKE or not p.endswith("_smoke.csv"))
chains = {}
for p in paths:
    lang, model = re.match(r"chain_([a-z]+)_(.+?)\.csv$", os.path.basename(p)).groups()
    df = pd.read_csv(p)
    df["bad"] = df.bad.fillna(False).astype(bool)
    chains[(lang, model)] = df
    have = [c for c in ["G", "L", "F", "F2", "B_L", "B_F"] if c in df]
    print(f"{lang} · {model}: {len(df)} items, {df.bad.sum()} flagged, columns {have}")
assert chains, f"no chain_*.csv files in {IN_DIR}"

# models must cover the same items to be compared item by item
ids = {k: set(d.item_id) for k, d in chains.items()}
common = set.intersection(*ids.values())
for k, s in ids.items():
    if s != common:
        print(f"⚠️ {k}: {len(s - common)} items not in every other file (cross-model tests use the {len(common)} shared items)")
items = pd.concat([d[["item_id", "domain", "source", "key", "S", "words"]] for d in chains.values()]).drop_duplicates("item_id")
items = items.sort_values(["domain", "source", "key"]).reset_index(drop=True)
DOMAINS = sorted(items.domain.unique())
print(len(items), "distinct items across", DOMAINS)

## 2 · Corpus: paraphrase ruler and Delta reference

In [ ]:
BASE = "https://raw.githubusercontent.com/tripto03/Ship_of_theseus_paraphrased_copus/main"
para, train = {}, {}
for d in DOMAINS:
    sot = pd.read_csv(f"{BASE}/paraphrased_datasets/{d}_paraphrased.csv")
    for name, version in PARAPHRASES.items():
        p = sot[sot.version_name == version].drop_duplicates(["key", "source"])
        for k, s, t in zip(p.key, p.source, p.text):
            para[(name, f"{d}|{s}|{k}")] = t
    train[d] = pd.read_csv(f"{BASE}/train_datasets/{d}_train.csv").text.dropna().tolist()
    print(f"{d}: {len(train[d])} train texts for the Delta reference")
for name in PARAPHRASES:
    items[name] = [para.get((name, i)) for i in items.item_id]
print("paraphrase coverage:", {n: int(items[n].notna().sum()) for n in PARAPHRASES})

## 3 · Metric code
Same definitions as `round_trip_stats.py`. Burrows' Delta z-scores the relative frequencies of the n most frequent words against the domain's unparaphrased `train` split.

In [ ]:
FUNCTION_WORDS = (
    "a an the of to in into on at for by with from about as and or but if than that because "
    "is are was were be been being has have had will would can could should may might must "
    "do does did not no i you he she it we they me him her us them my your his its our their "
    "this these those which who what there here also very more most").split()
TOKEN = re.compile(r"[a-z']+")

def funcword_profile(text):
    counts = collections.Counter(TOKEN.findall(text.lower()))
    n = max(1, len(text.split()))
    return np.array([counts[w] / n for w in FUNCTION_WORDS])

def sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]

def mean_sentence_length(text):
    return np.mean([len(s.split()) for s in sentences(text)] or [0])

def bigrams(text):
    w = re.findall(r"\w+", text.lower())
    return set(zip(w, w[1:]))

def numbers(text):
    return set(re.findall(r"\d[\d,.]*\d|\d", text))

def entities(text):
    # Capitalized words that are not sentence-initial: a cheap proxy for names.
    return {w for s in sentences(text) for w in re.findall(r"\b[A-Z][a-z]+", s)[1:]}

def cosine_distance(a, b):
    return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

def recall(ref, hyp):
    return len(ref & hyp) / len(ref) if ref else np.nan


class DeltaScorer:
    # z-scored most-frequent-word profiles for several vocabulary sizes, one reference per domain
    def __init__(self, reference_texts, sizes):
        total = collections.Counter(w for t in reference_texts for w in TOKEN.findall(str(t).lower()))
        self.mfw = [w for w, _ in total.most_common(max(sizes))]
        self.sizes = sizes
        ref = np.array([self.freqs(t) for t in reference_texts])
        self.mean, self.sd = ref.mean(0), ref.std(0) + 1e-12

    def freqs(self, text):
        c = collections.Counter(TOKEN.findall(str(text).lower()))
        n = max(1, sum(c.values()))
        return np.array([c[w] / n for w in self.mfw])

    def z(self, text):
        return (self.freqs(text) - self.mean) / self.sd

delta = {d: DeltaScorer(train[d], MFW_SIZES) for d in DOMAINS}


def metrics(x, t0, e_x, e_t0, z_x, z_t0):
    # Change from T0 to X. Higher = more changed, except the two recall metrics.
    b0 = bigrams(t0)
    out = dict(
        content=np.linalg.norm(e_x - e_t0),
        surface_chrF=1 - sacrebleu.sentence_chrf(x, [t0]).score / 100,
        surface_bigram=1 - len(b0 & bigrams(x)) / max(1, len(b0)),
        style_funcwords=cosine_distance(funcword_profile(x), funcword_profile(t0)),
        style_sentlen=abs(mean_sentence_length(x) - mean_sentence_length(t0)) / max(1e-9, mean_sentence_length(t0)),
        entity_recall=recall(entities(t0), entities(x)),
        number_recall=recall(numbers(t0), numbers(x)),
    )
    for n in MFW_SIZES:
        out[f"style_delta_{n}"] = np.mean(np.abs(z_x[:n] - z_t0[:n]))
        out[f"style_cosdelta_{n}"] = cosine_distance(z_x[:n], z_t0[:n])
    out["style_delta"], out["style_cosdelta"] = out[f"style_delta_{MFW_MAIN}"], out[f"style_cosdelta_{MFW_MAIN}"]
    return out

CHANGE = (["content", "surface_chrF", "surface_bigram", "style_delta", "style_cosdelta", "style_funcwords", "style_sentlen"]
          + [f"style_{k}_{n}" for k in ["delta", "cosdelta"] for n in MFW_SIZES])
RECALL = ["entity_recall", "number_recall"]

## 4 · Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer(EMB_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
_cache = {}

def embed(texts):
    # e5 embeddings, unit length; identical texts are embedded once across all models
    texts = [str(t) for t in texts]
    new = list({t for t in texts if t not in _cache})
    if new:
        for t, v in zip(new, emb_model.encode(["query: " + t for t in new], normalize_embeddings=True,
                                              batch_size=64, show_progress_bar=len(new) > 2000)):
            _cache[t] = v
    return np.array([_cache[t] for t in texts])

e0 = dict(zip(items.item_id, embed(items.S)))
z0 = {i: delta[d].z(s) for i, d, s in zip(items.item_id, items.domain, items.S)}
print(len(_cache), "texts embedded")

## 5 · Round-trip metrics and the paraphrase ruler
Random-pair baseline: each item against another item from the same domain × source, so normalized values are comparable across domains.

In [ ]:
def score_version(texts, ids):
    # metrics for one version (list of texts aligned with ids); NaN rows where the text is missing
    ok = [isinstance(t, str) and t.strip() != "" for t in texts]
    E = embed([t for t, k in zip(texts, ok) if k])
    rows, j = [], 0
    for i, t, k in zip(ids, texts, ok):
        if not k:
            rows.append({}); continue
        d = i.split("|", 1)[0]
        rows.append(metrics(t, S_of[i], E[j], e0[i], delta[d].z(t), z0[i])); j += 1
    return pd.DataFrame(rows, index=ids)

S_of = dict(zip(items.item_id, items.S))

# random-pair baseline (mean over items), per domain × source
partner = {}
for _, g in items.groupby(["domain", "source"]):
    gi = g.item_id.tolist()
    partner.update(zip(gi, np.roll(gi, -RANDOM_SHIFT)))
base_rows = score_version([S_of[partner[i]] for i in items.item_id], items.item_id.tolist())
base = base_rows[CHANGE].mean()
print("random-pair baseline (raw):", base[["content", "surface_chrF", "style_delta", "style_cosdelta"]].round(3).to_dict())

long = []
for name in PARAPHRASES:                              # ruler: computed once, model = "corpus"
    m = score_version(items[name].tolist(), items.item_id.tolist())
    long.append(m.assign(model="corpus", lang="en", version=name))
    print("scored", name)
for (lang, model), df in chains.items():
    for version, col in ROUND_TRIPS.items():
        if col in df:
            m = score_version(df[col].tolist(), df.item_id.tolist())
            long.append(m.assign(model=model, lang=lang, version=version))
            print("scored", model, version)
rt = pd.concat(long).rename_axis("item_id").reset_index()
for c in CHANGE:
    rt[f"{c}_norm"] = rt[c] / base[c]
rt = rt.merge(items[["item_id", "domain", "source", "words"]], on="item_id", how="left")

## 6 · Intervention (outgoing translation S1)
int_G(X) = ‖e(X) − e(G)‖ in target-language embedding space. `int_extra` = int_G(F) − int_G(L): the distance F travels beyond the literal translation on the same item.

In [ ]:
inter = []
for (lang, model), df in chains.items():
    if "G" not in df:
        print(f"{model} · {lang}: no gloss column, skipped"); continue
    out = df[["item_id", "bad"]].copy()
    EG = embed(df.G)
    for c in ["L", "F", "F2"]:
        if c not in df:
            continue
        has = df[c].notna().values
        E = np.full_like(EG, np.nan)
        E[has] = embed(df.loc[has, c])
        out[f"int_G_{c}"] = np.linalg.norm(E - EG, axis=1)
        out[f"chrF_G_{c}"] = [sacrebleu.sentence_chrf(str(x), [g]).score if h else np.nan
                              for x, g, h in zip(df[c], df.G, has)]
        out[f"_E_{c}"] = list(E)
    if "int_G_L" in out and "int_G_F" in out:
        out["int_extra"] = out.int_G_F - out.int_G_L
        out["d_LF"] = [np.linalg.norm(a - b) for a, b in zip(out._E_L, out._E_F)]
    if "int_G_F2" in out:
        out["noise_int_G"] = (out.int_G_F2 - out.int_G_F).abs()
        out["d_FF2"] = [np.linalg.norm(a - b) for a, b in zip(out._E_F, out._E_F2)]
    inter.append(out.drop(columns=[c for c in out if c.startswith("_E_")]).assign(model=model, lang=lang))
inter = pd.concat(inter) if inter else pd.DataFrame()

## 7 · Summary

In [ ]:
def paired(a, b, less=True):
    # paired Wilcoxon on items present in both; returns medians, share of items in the stated direction, p
    j = pd.concat([a, b], axis=1, keys=["a", "b"]).dropna()
    if len(j) < 6:
        return dict(n=len(j), a=np.nan, b=np.nan, share=np.nan, p=np.nan)
    share = (j.a < j.b).mean() if less else (j.a > j.b).mean()
    return dict(n=len(j), a=j.a.median(), b=j.b.median(), share=share, p=wilcoxon(j.a, j.b).pvalue)

clean = {(l, m): set(d.item_id[~d.bad]) for (l, m), d in chains.items()}
summary = {}
pd.set_option("display.width", 200)

for (lang, model) in chains:
    ok = clean[(lang, model)]
    S = {}
    # intervention
    iv = inter[(inter.model == model) & (inter.lang == lang) & inter.item_id.isin(ok)].set_index("item_id") if len(inter) else pd.DataFrame()
    if len(iv):
        S["intervention"] = {c: float(iv[c].median()) for c in iv
                              if c.startswith(("int_G", "int_extra", "noise", "d_")) and iv[c].notna().any()}
        if "int_extra" in S["intervention"]:
            S["F_vs_L"] = paired(iv.int_G_F, iv.int_G_L, less=False)
            S["LF_vs_noise"] = paired(iv.d_LF, iv.d_FF2, less=False) if "d_FF2" in iv else None
    # round trip vs ruler (normalized medians), on this model's clean items
    sub = rt[rt.item_id.isin(ok) & ((rt.model == model) | (rt.model == "corpus"))]
    norm = sub.groupby("version")[[f"{c}_norm" for c in ["content", "surface_chrF", "surface_bigram", "style_delta", "style_cosdelta", "style_funcwords", "style_sentlen"]]].median()
    order = [v for v in list(ROUND_TRIPS) + list(PARAPHRASES) if v in norm.index]
    S["normalized"] = norm.loc[order].round(4).to_dict("index")
    # T1 free vs ChatGPT ×1, raw, paired
    t1 = sub[sub.version == "T1 free"].set_index("item_id")
    cg = sub[sub.version == "chatgpt×1"].set_index("item_id")
    if len(t1):
        S["t1_vs_chatgpt"] = {m: paired(t1[m], cg[m], less=m in CHANGE) for m in CHANGE[:7] + RECALL}
        # style ÷ content within a version, by vocabulary size
        S["style_over_content"] = {v: {n: (g[f"style_delta_{n}_norm"] / g.content_norm).median() for n in MFW_SIZES}
                                   for v, g in [("T1 free", t1), ("chatgpt×1", cg)]}
        S["content_by_source"] = t1.groupby("source").content.median().round(4).to_dict()
    summary[f"{lang}|{model}"] = S

    print(f"\n══ {model} · {lang} · {len(ok)} clean items ══")
    if "intervention" in S:
        print("intervention medians:", {k: round(v, 4) for k, v in S["intervention"].items()})
        if "F_vs_L" in S:
            r = S["F_vs_L"]; print(f"  int_G(F) > int_G(L): {r['share']:.0%} of items, p = {r['p']:.2g}")
    print("normalized change from T0 (0 = unchanged, 1 = unrelated text):")
    display(pd.DataFrame(S["normalized"]).T.round(3))
    if "t1_vs_chatgpt" in S:
        display(pd.DataFrame(S["t1_vs_chatgpt"]).T.rename(columns={"a": "T1 free", "b": "chatgpt×1",
                                                                   "share": "T1 less change / more kept in"}).round(4))
        display(pd.DataFrame(S["style_over_content"]).round(3).rename_axis("MFW"))

### Across models
Only when more than one model is loaded. Paired by item on items that are clean for both models. `metric ~ model + (1 | item)` mixed-effects fits use statsmodels.

In [ ]:
models = [m for (l, m) in chains]
if len(models) > 1:
    import statsmodels.formula.api as smf
    both = set.intersection(*clean.values())
    print(len(both), "items clean for every model")
    cross = {}
    for metric in ["content", "style_delta", "surface_chrF"]:
        d = rt[(rt.version == "T1 free") & rt.item_id.isin(both) & (rt.model != "corpus")][["item_id", "model", metric]].dropna()
        fit = smf.mixedlm(f"{metric} ~ C(model)", d, groups=d.item_id).fit()
        cross[metric] = {k: (round(v, 5), fit.pvalues[k]) for k, v in fit.fe_params.items()}
        print(f"\n{metric} (T1 free):"); print(fit.summary().tables[1])
    if len(inter) and "int_extra" in inter and inter.groupby("model").int_extra.count().gt(0).all():
        d = inter[inter.item_id.isin(both)][["item_id", "model", "int_G_F", "int_extra"]].dropna()
        for metric in ["int_G_F", "int_extra"]:
            fit = smf.mixedlm(f"{metric} ~ C(model)", d, groups=d.item_id).fit()
            cross[metric] = {k: (round(v, 5), fit.pvalues[k]) for k, v in fit.fe_params.items()}
            print(f"\n{metric}:"); print(fit.summary().tables[1])
    summary["across_models"] = cross
else:
    print("one model loaded — add another chain_*.csv to IN_DIR for the comparison")

## 8 · Save

In [ ]:
rt.to_csv(f"{OUT_DIR}/metrics_items.csv", index=False)                 # one row per model × item × T1 / paraphrase version
if len(inter):
    inter.to_csv(f"{OUT_DIR}/metrics_intervention.csv", index=False)  # one row per model × item
json.dump(summary, open(f"{OUT_DIR}/metrics_summary.json", "w"), indent=1, default=float)
print("saved metrics_items.csv", rt.shape, "| metrics_intervention.csv", inter.shape, "| metrics_summary.json")

## 9 · Quality estimate (COMETKiwi, optional)
Same isolated setup as `degree_zero_qe.ipynb`: COMET lives in its own Python 3.11 environment because `unbabel-comet` pins old `numpy` / `huggingface-hub`. Needs the `Unbabel/wmt22-cometkiwi-da` license accepted on Hugging Face and `HF_TOKEN` in Colab Secrets.

COMETKiwi reads source and translation together, capped at 512 tokens, so long items are truncated and QE falls with length. Compare conditions and models on the same items, not items with each other.

In [ ]:
if RUN_QE:
    !pip install -q uv
    !uv venv /content/qe_env --python 3.11 -q
    !uv pip install -q --python /content/qe_env/bin/python unbabel-comet pandas "setuptools<81"
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

    rows = []
    for (lang, model), df in chains.items():
        for c in ["L", "F", "F2"]:
            if c in df:
                for i, s, t in zip(df.item_id, df.S, df[c]):
                    if isinstance(t, str) and t.strip():
                        rows.append(dict(lang=lang, model=model, cond=c, item_id=i, src=s, mt=t))
    pd.DataFrame(rows).to_csv("/content/qe_in.csv", index=False)
    open("/content/score_qe.py", "w").write('''
import os, sys, pandas as pd
from huggingface_hub import login
from comet import download_model, load_from_checkpoint
login(token=os.environ["HF_TOKEN"])
df = pd.read_csv(sys.argv[1])
model = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))
df["qe"] = model.predict([{"src": s, "mt": m} for s, m in zip(df.src, df.mt)],
                         batch_size=16, gpus=1, progress_bar=True).scores
df.drop(columns=["src", "mt"]).to_csv(sys.argv[2], index=False)
''')
    print(len(rows), "translations to score")

In [ ]:
if RUN_QE:
    !/content/qe_env/bin/python /content/score_qe.py /content/qe_in.csv {OUT_DIR}/qe_scores.csv
    qe = pd.read_csv(f"{OUT_DIR}/qe_scores.csv")
    for (lang, model), g in qe.groupby(["lang", "model"]):
        w = g.pivot(index="item_id", columns="cond", values="qe")
        w = w[w.index.isin(clean[(lang, model)])]
        print(f"\n{model} · {lang}: median QE", w.median().round(3).to_dict())
        if {"L", "F"} <= set(w):
            r = paired(w.F, w.L, less=False); print(f"  QE(F) > QE(L): {r['share']:.0%} of items, p = {r['p']:.2g}")
        if {"F", "F2"} <= set(w):
            r = paired(w.F, w.F2, less=False); print(f"  QE(F) vs QE(F2) [noise]: median diff {(w.F - w.F2).median():+.4f}, p = {r['p']:.2g}")